## Advanced SQL Assignment: Student Submission Analysis

> **File paths (Databricks Volumes)**
> - Master: `/Volumes/workspace/default/day4/Capgemini Databricks VITB-2026 (1).csv`
> - Task1 Responses: `/Volumes/workspace/default/day4/Task 1 (Responses).csv`
> - Task1 File2: `/Volumes/workspace/default/day4/Task 1 file 2.csv`

## Phase 1: Load & Normalize Data

#### Load master table (56 students)

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW students
USING CSV
OPTIONS (
  path "/Volumes/workspace/default/day4/Capgemini Databricks VITB-2026 (1).csv",
  header "true",
  inferSchema "true"
)


#### Load Task1 Responses (51 records)

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW task1_responses
USING CSV
OPTIONS (
  path "/Volumes/workspace/default/day4/Task 1 (Responses).csv",
  header "true",
  inferSchema "true"
)


#### Load Task1 File2 (60 records - has duplicates + invalid + extra)

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW task1_file2
USING CSV
OPTIONS (
  path "/Volumes/workspace/default/day4/Task 1 file 2.csv",
  header "true",
  inferSchema "true"
)


#### Normalize master table: lowercase/trim emails, alias columns

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW master AS
SELECT
    `REGNO`                                    AS student_id,
    `STUDENT NAME`                             AS student_name,
    LOWER(TRIM(`College Domain MAILID`))       AS college_email,
    LOWER(TRIM(`Personal Mail ID`))            AS personal_email
FROM students
WHERE `REGNO` IS NOT NULL


In [0]:
%sql
SELECT * FROM master LIMIT 5


#### Normalize Task1 Responses: extract only valid email rows, clean timestamp

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW responses_clean AS
SELECT
    TRY_TO_TIMESTAMP(`Timestamp`, 'M-d-yyyy H:mm:ss') AS submission_time,
    LOWER(TRIM(`Email address`))                        AS email,
    TRIM(`Have you completed all the 5 queries? `)      AS completed_queries,
    TRIM(`  GitHub Username  `)                         AS github_username,
    TRIM(`Repository Link `)                            AS repo_link
FROM task1_responses
WHERE `Email address` IS NOT NULL
  AND TRIM(`Email address`) != ''
  AND LOWER(TRIM(`Email address`)) LIKE '%@%'


#### Normalize Task1 File2: same cleaning, different timestamp format

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW file2_clean AS
SELECT
    TRY_TO_TIMESTAMP(`Timestamp`, 'dd-MM-yyyy HH:mm') AS submission_time,
    LOWER(TRIM(`Email address`))                       AS email,
    TRIM(`Have you completed all the 5 queries? `)     AS completed_queries,
    TRIM(`  GitHub Username  `)                        AS github_username,
    TRIM(`Repository Link `)                           AS repo_link
FROM task1_file2
WHERE `Email address` IS NOT NULL
  AND TRIM(`Email address`) != ''
  AND LOWER(TRIM(`Email address`)) LIKE '%@%'


#### Phase 1.2: Unified email mapping — both emails → student_id

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW email_mapping AS
SELECT student_id, college_email AS email FROM master WHERE college_email IS NOT NULL
UNION
SELECT student_id, personal_email AS email FROM master WHERE personal_email IS NOT NULL


In [0]:
%sql
SELECT * FROM email_mapping LIMIT 10


## Phase 2: Combine Both Submission Files

#### UNION both files to get all raw submissions (before dedup/validation)

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW all_submissions_raw AS
SELECT *, 'file1' AS source FROM responses_clean
UNION ALL
SELECT *, 'file2' AS source FROM file2_clean


#### Phase 2.1: VALID submissions — emails found in master (INNER JOIN)

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW valid_submissions AS
SELECT
    em.student_id,
    s.email,
    s.submission_time,
    s.completed_queries,
    s.github_username,
    s.repo_link,
    s.source
FROM all_submissions_raw s
INNER JOIN email_mapping em
ON s.email = em.email


In [0]:
%sql
SELECT COUNT(*) AS valid_count FROM valid_submissions


#### Phase 2.2: INVALID submissions — emails NOT in master

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW invalid_submissions AS
SELECT s.email, s.submission_time, s.github_username, s.source
FROM all_submissions_raw s
LEFT ANTI JOIN email_mapping em
ON s.email = em.email


In [0]:
%sql
SELECT * FROM invalid_submissions


#### Phase 2.3: NOT SUBMITTED — students in master with no matching submission

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW not_submitted AS
SELECT m.student_id, m.student_name, m.college_email
FROM master m
LEFT ANTI JOIN valid_submissions vs
ON m.student_id = vs.student_id


In [0]:
%sql
SELECT * FROM not_submitted ORDER BY student_id


In [0]:
%sql
SELECT COUNT(*) AS not_submitted_count FROM not_submitted


## Phase 3: Duplicate Detection (Window Functions)

#### ROW_NUMBER() — rank submissions per student (earliest first); rn=1 = first, rn>1 = duplicate

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW deduped AS
SELECT *,
    ROW_NUMBER() OVER (
        PARTITION BY student_id
        ORDER BY submission_time ASC
    ) AS rn
FROM valid_submissions


#### First (unique) submissions per student

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW unique_submissions AS
SELECT * FROM deduped WHERE rn = 1


#### Duplicate submissions (same student submitted more than once)

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW duplicates AS
SELECT * FROM deduped WHERE rn > 1


In [0]:
%sql
SELECT student_id, email, submission_time, rn FROM deduped ORDER BY student_id, rn


#### Compare: GROUP BY approach (loses row-level detail)

In [0]:
%sql
SELECT student_id, COUNT(*) AS total_submissions
FROM valid_submissions
GROUP BY student_id
HAVING COUNT(*) > 1


## Phase 4: Advanced Insights

#### 4.1: Submission count per student

In [0]:
%sql
SELECT student_id, COUNT(*) AS total_submissions
FROM valid_submissions
GROUP BY student_id
ORDER BY total_submissions DESC


#### 4.2: Students who submitted using BOTH emails (college + personal)

In [0]:
%sql
SELECT student_id, COUNT(DISTINCT email) AS emails_used
FROM valid_submissions
GROUP BY student_id
HAVING COUNT(DISTINCT email) > 1


#### 4.3: Final classification — Submitted / Duplicate / Not Submitted / Invalid

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW final_classification AS

-- Valid first-time submitters
SELECT student_id, email, submission_time, completed_queries, github_username, repo_link,
       'Submitted'     AS status
FROM unique_submissions

UNION ALL

-- Students who submitted more than once (keep the extras marked as duplicate)
SELECT student_id, email, submission_time, completed_queries, github_username, repo_link,
       'Duplicate'     AS status
FROM duplicates

UNION ALL

-- Students who never submitted
SELECT student_id, NULL AS email, NULL AS submission_time,
       NULL AS completed_queries, NULL AS github_username, NULL AS repo_link,
       'Not Submitted'  AS status
FROM not_submitted


#### Final dashboard — every student classified

In [0]:
%sql
SELECT * FROM final_classification ORDER BY status, student_id


#### Summary counts

In [0]:
%sql
SELECT status, COUNT(*) AS count
FROM final_classification
GROUP BY status
ORDER BY count DESC


#### Invalid submissions (emails not in master at all)

In [0]:
%sql
SELECT email, submission_time, github_username, source, 'Invalid' AS status
FROM invalid_submissions
ORDER BY email
